<a href="https://colab.research.google.com/github/FilipeSchweitzer/ProjetoAplicado2_CDIA_UniSenai/blob/main/ist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df_ident = pd.read_excel('IDENTIFICACAO.xlsx', usecols=['Compound', 'Compound ID', 'Formula', 'Fragmentation Score', 'Score', 'Isotope Similarity', 'Description'])
df_abund = pd.read_excel('ABUND.xlsx', usecols=['Compound', 'Identifications'])

df = pd.merge(df_ident, df_abund, on='Compound')
df = df.dropna(subset='Description')

csv = df.to_csv('table.csv', index=False)

df = pd.read_csv('table.csv')

In [ ]:
import requests
import json

In [ ]:


def Pubchem_search(name:str):
    result = {}

    main = Pubchem_main(name)
    taxonomy = Pubchem_taxonomy(name)

    for key, value in main.items():
        result[key] = value
    for key, value in taxonomy.items():
        result[key] = value

    return result

def Pubchem_main(name:str):
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/JSON'

    response = requests.get(url, timeout=10)
    code = response.status_code

    if code == 200:
        data = response.json()
        props = data['PC_Compounds'][0]['props']
        return {p['urn']['label']: p['value'] for p in props
                if p['urn']['label'] in ['IUPAC Name', 'SMILES', 'InChIKey']}
    else:
        if code != 404:
            update_error_log(f'{name} teve o erro: {code}')
        return {}

def Pubchem_taxonomy(name:str):
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/description/JSON'

    response = requests.get(url, timeout=10)
    code = response.status_code

    if code == 200:
        data = response.json() #devolve 2 dicts um com o cids e outro com as descrições
        data_list = data['InformationList']['Information']

        for item in data_list: #itera sob os 2 dicionários, se tiver a descrição, é esse dict que queremos
            if 'Description' in item:
                desc = item.get('Description', '')
                desc_src = item.get('DescriptionSourceName', '')
                break

        return {'Information': desc, 'Info_Source' : desc_src}
    else:
        if code != 404:
            update_error_log(f'{name} teve o erro: {code}')
        return {}

# print(Pubchem_search('Adozelesin'))

In [ ]:
tabela_final = pd.DataFrame(columns=['InChIKey', 'Compound ID', 'Description', 'IUPAC Name', 'Formula', 'SMILES', 'Information',
                                     'Info_Source', 'Fragmentation Score', 'Score', 'Isotope Similarity', 'Identifications'])
print(tabela_final.columns)

def add_to_final(index, info):
    original_table = df.iloc[index]
    api_search = info

    # print(original_table)

    data = []
    for column in tabela_final.columns:
        if column in original_table.index:
            #print(f'{column}: {original_table.loc[column]}')
            data.append(original_table[column])
        elif column in info.keys():
            if 'sval' in info[column]:
                #print(f'{column}: {info[column]['sval']}')
                data.append(info[column]['sval'])
            else:
                data.append(info[column])
        else:
            data.append('Nan')

    print(data)

    # adiciona os valores a tabela final
    if len(data) == len(tabela_final.columns):
        tabela_final.loc[len(tabela_final)] = data
    else:
        print(f'\nHÁ COLUNAS EXTRAS !!!\n {data}\n{tabela_final.columns}')

Index(['Compound ID', 'Description', 'IUPAC Name', 'Formula', 'SMILES',
       'Information', 'Info_Source', 'Fragmentation Score', 'Score',
       'Isotope Similarity', 'Identifications'],
      dtype='object')


In [ ]:
def create_error_log():
    with open('error_log.txt', 'w'):
        pass

def update_error_log(error):
    with open('error_log.txt', 'a') as log:
        log.write(f'\n{error}')

In [ ]:
#exvazia a tabela
def clear_table():
    import os
    global tabela_final
    tabela_final = tabela_final.iloc[0:0]

    arquivo = 'final.csv'
    if os.path.exists(arquivo):
        os.remove(arquivo)

clear_table()

MAIN

In [ ]:
clear_table()
create_error_log()

from time import sleep

found = 0
print(tabela_final.columns)
for index in range(50, 60):
    try:
        compound = df.iloc[index]['Description']

        search = Pubchem_search(compound)

        if search != 'Error: 404':
            found += 1
            print(f'{index}.{compound} / found:{found}')
            # print(search)
            add_to_final(index, search)
    except Exception as e:
        update_error_log(f"ERRO!!! índice:{index}, erro:{e}")
        continue
    sleep(0.2)

#tabela_final.to_excel('final.xlsx', index=False)
tabela_final.to_csv('final.csv', index=False)
print('finished')

Index(['InChIKey', 'Compound ID', 'Description', 'IUPAC Name', 'Formula',
       'SMILES', 'Information', 'Info_Source', 'Fragmentation Score', 'Score',
       'Isotope Similarity', 'Identifications'],
      dtype='object')
50.Adozelesin / found:1
['BYRVKDUQDLJUBX-JJCDCTGGSA-N', 'CSID16736561', 'Adozelesin', 'N-[2-[(1R,12S)-7-keto-3-methyl-5,10-diazatetracyclo[7.4.0.01,12.02,6]trideca-2(6),3,8-triene-10-carbonyl]-1H-indol-5-yl]coumarilamide', 'C30H22N4O4', 'CC1=CNC2=C1C34CC3CN(C4=CC2=O)C(=O)C5=CC6=C(N5)C=CC(=C6)NC(=O)C7=CC8=CC=CC=C8O7', 'Adozelesin is an indolecarboxamide.', 'ChEBI', np.float64(0.0), np.float64(37.0), np.float64(85.6385831456806), np.int64(11)]
51.(1R,2S,3R)-2-(2,4-Dihydroxyphenyl)-3-(3,5-dihydroxyphenyl)-1-[(R)-(4-hydroxyphenyl)(methoxy)methyl]-4,6-indanediol / found:2
['Nan', 'CSID28288545', '(1R,2S,3R)-2-(2,4-Dihydroxyphenyl)-3-(3,5-dihydroxyphenyl)-1-[(R)-(4-hydroxyphenyl)(methoxy)methyl]-4,6-indanediol', 'Nan', 'C29H26O8', 'Nan', 'Nan', 'Nan', np.float64(0.0), np.